In [1]:
import numpy as np
from Bio import PDB
from Bio.PDB import PDBParser


In [3]:
'''Step 1: Counting n-neighbours for atoms'''

def convert_cif_to_pdb(cif_file_path, pdb_file_path):
    '''
    Takes in a .cif file, and returns a .pdb file.

    Both file paths should be specified below    
    '''

    parser = PDB.MMCIFParser()
    structure = parser.get_structure('protein', cif_file_path)

    io = PDB.PDBIO()
    io.set_structure(structure)
    io.save(pdb_file_path)

def extract_pdb_info(pdb_file_path):
    """
    Reads a PDB file and extracts 3D spatial coordinates and amino acids.

    Input: pdb_file_path (str): Path to the PDB file.

    Output:  A list of dictionaries, each containing information about each atom.
    """
    # Create a PDBParser object
    parser = PDBParser(PERMISSIVE=1)
    
    # Parse the structure from the PDB file
    structure = parser.get_structure('protein', pdb_file_path)
    
    # List to hold the extracted information
    atom_info_list = []

    # Extract information from the structure
    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    atom_info = {
                        'model_id': model.id,
                        'chain_id': chain.id,
                        'residue_name': residue.resname,
                        'residue_id': residue.id[1],
                        'atom_name': atom.name,
                        'atom_coords': atom.coord.tolist()
                    }
                    atom_info_list.append(atom_info)

    return atom_info_list


def count_neighboring_atoms(data):
    '''
    Adds 'neighbour_count': n for each atom data dictionary

    Input: A list of dictionaries, each containing information about each atom.
    
    Output: The same list of dictionaries, appended with 'neighbour_count': n for each atom data dictionary
    '''

    def distance_np(coord1, coord2):
        return np.linalg.norm(coord1 - coord2)

    def atom_nearest_neighbours(data):

        coords = np.array([atom['atom_coords'] for atom in data])
        neighbour_counts = np.zeros(len(data), dtype=int)
        
        for i in range(len(coords)):
            for j in range(i+1, len(coords)):
                distances = distance_np(coords[i],coords[j])
                if distances <= 3:
                    neighbour_counts[i] += 1

        for i in range(len(neighbour_counts)):
            data[i]['neighbour_count'] = neighbour_counts[i]
        print(neighbour_counts)
        
        return data

    neighbour_counts = atom_nearest_neighbours(data)

    '''
    #delete irrelevant ids in data if required at this step.
    for entry in range(len(neighbour_counts)):
        del neighbour_counts[entry]['model_id']
        del neighbour_counts[entry]['chain_id']
        # del neighbour_counts[entry]['residue_name']
        del neighbour_counts[entry]['residue_id']
    '''
        
    return neighbour_counts

In [4]:
'''Step 2: Counting n-neighbours for amino acids'''

def count_neighbouring_amino_acids(data):
    '''
    Trim atom dictionary to retain one entry per amino acid with the highest neighbour_count

    Takes in dictionary of atom data appended with key-value pair 'neighbour_count': n  

    Returns dictionary of amino acids with key-value pair 'neighbour_count': n 
    '''

    # Dictionary to store the highest 'neighbour_count' for each 'residue_id'
    max_neighbour_counts = {}

    # Iterate through the data to find the highest 'neighbour_count' for each 'residue_id'
    for atom in data:
        residue_id = atom['residue_id']
        neighbour_count = atom.get('neighbour_count', 0)
        if residue_id not in max_neighbour_counts or neighbour_count > max_neighbour_counts[residue_id]:
            max_neighbour_counts[residue_id] = neighbour_count

    # Filter the original data to retain only one entry for each 'residue_id' with the highest 'neighbour_count'
    filtered_data = []
    for atom in data:
        residue_id = atom['residue_id']
        neighbour_count = atom.get('neighbour_count', 0)
        if neighbour_count == max_neighbour_counts[residue_id]:
            # Remove other entries for the same residue_id
            filtered_data = [entry for entry in filtered_data if entry['residue_id'] != residue_id]
            filtered_data.append(atom)
            atom.pop('model_id')
            atom.pop('atom_name')
            atom.pop('atom_coords')

    return filtered_data


In [6]:
# ============== MAIN =======================

cif_file_path = './7sfc_updated.cif'
pdb_file_path = './pdb7sfc.pdb'

'''Step 1: Counting n-neighbours for atoms'''
convert_cif_to_pdb(cif_file_path, pdb_file_path)
atom_info_list = extract_pdb_info(pdb_file_path)
print(len(atom_info_list))

num_atom_entries = 20  #Adjust this, total is 7714 atoms
atom_info_count = count_neighboring_atoms(atom_info_list[:num_atom_entries])

#Check loading
for atom_info in atom_info_count:
    print(atom_info)

'''Step 2: Counting n-neighbours for amino acids'''
amino_acid_info_count = count_neighbouring_amino_acids(atom_info_count)

#Check amino_acid_neighbour, should have 1389
num_amino_acid_entries = 20 #Adjust this
for amino_acid in amino_acid_info_count[:num_amino_acid_entries]:  
    print(f"{amino_acid}")

print(f"Number of Amino Acids: {len(amino_acid_info_count)}")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 7062.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 7220.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 7254.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 7747.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain D is discontinuous at line 7752.
  warnings.w

7714
[4 5 4 ... 0 0 0]
{'model_id': 0, 'chain_id': 'A', 'residue_name': 'ARG', 'residue_id': 730, 'atom_name': 'N', 'atom_coords': [24.583999633789062, -18.43199920654297, 42.12099838256836], 'neighbour_count': 4}
{'model_id': 0, 'chain_id': 'A', 'residue_name': 'ARG', 'residue_id': 730, 'atom_name': 'CA', 'atom_coords': [24.886999130249023, -18.79400062561035, 40.742000579833984], 'neighbour_count': 5}
{'model_id': 0, 'chain_id': 'A', 'residue_name': 'ARG', 'residue_id': 730, 'atom_name': 'C', 'atom_coords': [24.10700035095215, -17.91200065612793, 39.75600051879883], 'neighbour_count': 4}
{'model_id': 0, 'chain_id': 'A', 'residue_name': 'ARG', 'residue_id': 730, 'atom_name': 'O', 'atom_coords': [23.56399917602539, -18.391000747680664, 38.750999450683594], 'neighbour_count': 4}
{'model_id': 0, 'chain_id': 'A', 'residue_name': 'ARG', 'residue_id': 730, 'atom_name': 'CB', 'atom_coords': [24.584999084472656, -20.273000717163086, 40.5099983215332], 'neighbour_count': 2}
{'model_id': 0, 'ch